# Notebook 02: Medical Image Preprocessing and Exploratory Data Analysis (EDA)
## Hallucination-Resistant Medical Report Generation Using Generative AI

### Research Objectives:
1. Analyze class distributions across target abnormalities (Cardiomegaly, Effusion, Pneumothorax, etc.).
2. Perform image quality audits (dimensions, aspect ratios, dynamic range, contrast statistics).
3. Demonstrate DICOM and standard image format ingestion.
4. Build and visualize a multi-stage preprocessing pipeline (Original -> Resized -> Normalized -> CLAHE).


In [ ]:
# 1. Module Imports and Environment Check
import sys
from pathlib import Path
for p in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/hallucination_medical_ai')]:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import pandas as pd
import matplotlib.pyplot as plt
from src.config import get_config
from src.preprocessing.preprocessor import MedicalImagePreprocessor
from src.visualization.plots import plot_class_distributions, plot_preprocessing_comparison

cfg = get_config()
print('Loaded configuration successfully.')


### 2. Load Indexed Dataset and Analyze Abnormality Distribution


In [ ]:
index_path = cfg.output_dir / 'metrics' / 'dataset_index.csv'
if index_path.exists():
    df = pd.read_csv(index_path)
    print(f'Loaded {len(df)} indexed samples from {index_path}')
else:
    from src.datasets.discovery import DatasetDiscovery
    df = DatasetDiscovery(cfg).build_unified_index(max_samples=100)

# Visualize class frequencies
fig = plot_class_distributions(df, cfg.dataset.get('abnormality_classes', []))
plt.show()


### 3. Medical Image Quality Audits
Detect low-contrast scans, unusual aspect ratios, or low resolution that could trigger visual hallucination in downstream generators.


In [ ]:
from src.preprocessing.quality import assess_image_quality

sample_img_path = df['image_path'].iloc[0] if not df.empty else 'data/sample/sample_cxr.png'
q_metrics = assess_image_quality(sample_img_path)
print('Quality Assessment Results:')
for k, v in q_metrics.items():
    print(f'  {k}: {v}')


### 4. Image Preprocessing Pipeline and Visual Comparison


In [ ]:
preprocessor = MedicalImagePreprocessor(target_size=(224, 224), apply_clahe=False)
processed_tensor, stages = preprocessor.process_with_intermediate_stages(sample_img_path)

fig = plot_preprocessing_comparison(stages)
plt.show()
